Workflow plan:

- Takes an area geojson, like example_datasets/Leeds_pp_or_g_cmb.geojson
- Checks if it's Wales or Leeds using the LUT desaigned in data_structuring repository (example_datasets/all_parks_ids.csv)
- Uses appropriate workflow (Wales vs England)
- Outputs all data in a folder withy the same prefix as the area geojson
- All datasets have the unique park id from the LUT as a prefix to their name


In [1]:
import geopandas as gpd
import json
from pathlib import Path
import sys
import pandas as pd
import matplotlib.pyplot as plt


sys.path.insert(0, '../src')

import park_vga

In [2]:
boundaries_folder = "../example_datasets/"
boundaries_filename = "Leeds_pp_or_g_cmb.geojson"
boundaries_file = Path(boundaries_folder) / boundaries_filename

eng_dtm = "/Volumes/Extreme SSD/DTM"
eng_dsm = "/Volumes/Extreme SSD/FZ_DSM"
wales_dtm = "/Volumes/Extreme SSD/wales_lidar/wales_dtm_32bit_cog.tif"
wales_dsm = "/Volumes/Extreme SSD/wales_lidar/wales_dsm_32bit_cog.tif"

output_folder = "../workflow_outputs/"

check_regions_file = "../example_datasets/LUT_regions_authorities_filenames.geojson"
park_ids_file = "../example_datasets/all_parks_ids.csv"

In [3]:
# load the boundaries file as a geodataframe
gdf = gpd.read_file(boundaries_file)

# pull out the title of the boundaries file (after last "/" and before ".geojson")
boundaries_title = boundaries_filename.split(".")[0]
print(boundaries_title)

output_path = Path(output_folder) / boundaries_title
print(output_path)
#check that output path exists, and if not, create it
if not output_path.exists():
    output_path.mkdir(parents=True, exist_ok=True)

# use check_regions_file  to check country for the boundaries file
# see if [filename] matches boundaries_filename in check_regions_file, and if so, pull out the country
check_regions_gdf = gpd.read_file(check_regions_file)
check_regions_gdf["filename"] = check_regions_gdf["filename"].apply(lambda x: Path(x).name)
country = check_regions_gdf.loc[check_regions_gdf["filename"] == boundaries_filename, "country"].values[0]
authority = check_regions_gdf.loc[check_regions_gdf["filename"] == boundaries_filename, "auth_name_e"].values[0]
print(country, authority)


# filter the park ids dataframe to just the authority
park_ids_file_df = pd.read_csv(park_ids_file)
# subset to where auth_name_e matches authority
park_ids_file_df = park_ids_file_df.loc[park_ids_file_df["auth_name_e"] == authority]
print(len(park_ids_file_df))


Leeds_pp_or_g_cmb
../workflow_outputs/Leeds_pp_or_g_cmb
England Leeds
1058


In [4]:
park_ids_file_df

,country,region_id,authority_id,auth_name_e,old_park_id,new_park_id
118086,England,E12000003,E08000035,Leeds,LEEDS_8192062f3fdc,e27de60bcc90
118087,England,E12000003,E08000035,Leeds,LEEDS_9c93a531629b;LEEDS_c315f5a3beb9,34fa605194c3
118088,England,E12000003,E08000035,Leeds,LEEDS_61d3fdd3c28e;LEEDS_f05b1d7906e8,bc1dd516d190
118089,England,E12000003,E08000035,Leeds,LEEDS_8ec1a7840269;LEEDS_9648dc192d18,52a600310c20
118090,England,E12000003,E08000035,Leeds,LEEDS_7517d5eed785,a3f253420457
...,...,...,...,...,...,...
119139,England,E12000003,E08000035,Leeds,LEEDS_1631c5f8e4c8,b654b4e49f74
119140,England,E12000003,E08000035,Leeds,LEEDS_7baf06d10dfc,c3a88bbf3432
119141,England,E12000003,E08000035,Leeds,LEEDS_2f568f82ebf9,48e2819aabd2
119142,England,E12000003,E08000035,Leeds,LEEDS_f28f723b9add,2f028bfd7f48


In [6]:
for n in range(len(gdf)):
    print(f"{n}/{len(gdf)}")
    park_id = gdf.loc[n, "id"]
    print(park_id)
    # use subsetted park ids dataframe to find the row where park_id matches id in park_ids_file_df
    # and pull out new park id (safe for filenames)
    park_id_safe = park_ids_file_df.loc[park_ids_file_df["old_park_id"] == park_id, "new_park_id"].values[0]
    print(park_id_safe)
    # check if output files already exist for this park, and if so, skip to the next park
    output_file = output_path / f"{park_id_safe}_visibility.geojson"
    if output_file.exists():
        print(f"Output file {output_file} already exists, skipping park {park_id_safe}")
        continue

    if country == "England":
        dtm_path = eng_dtm
        dsm_path = eng_dsm
        results = park_vga.workflow.workflow_eng(boundaries_file, n,
                                            dtm_path, dsm_path,
                                            output_path,
                                            spacing=12,
                                            return_results=False, save_results=True,
                                            park_id_for_file_name=park_id_safe,
                                            max_distance=80)


    elif country == "Wales":
        dtm_path = wales_dtm
        dsm_path = wales_dsm


0/1058
LEEDS_8192062f3fdc
e27de60bcc90
Output directory already exists at ../workflow_outputs/Leeds_pp_or_g_cmb
Park ID: LEEDS_8192062f3fdc
Park ID for file name: e27de60bcc90
Define park grid.
Load park tiles
Finding tile names from park geometry
Calculating grid refs
[(422799.7595083842, 433163.1205998056), (422799.7595083842, 433296.48116122687), (422614.766435792, 433296.48116122687), (422614.766435792, 433163.1205998056)]
Grid refs for corners: ['SE23sw', 'SE23sw', 'SE23sw', 'SE23sw']
Tile name(s): {'SE23sw'} 
Finding file paths for these tiles
Paths found: 
DTM paths: ['/Volumes/Extreme SSD/DTM/SE23sw_DTM_2m.tif'] 
DSM paths: ['/Volumes/Extreme SSD/FZ_DSM/SE23sw_FZ_DSM_2m.tif']
Files found for park 1
Created 122 pointy-top hexagons (12m vertical spacing)
Loading 1 DSM and 1 DTM files
DSM valid pixels: 6250000 of 6250000 (100.0%)
DTM valid pixels: 6250000 of 6250000 (100.0%)
Output shape: (68, 93), Height range: -0.1m to 16.3m
Start calculating visibility. This will take a few min